# setup + imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import f1_score

In [2]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [3]:
Xcols = ['id', 'timestamp', 'server_id', 'region', 'server_type', 'cpu_usage', 'memory_usage', 'disk_io', 'network_latency', 'error_rate', 'temperature', 'health_score', 'alert_title', 'system_log']

# data exploration

In [4]:
train_data.info()  

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               6000 non-null   int64  
 1   timestamp        6000 non-null   str    
 2   server_id        6000 non-null   str    
 3   region           6000 non-null   str    
 4   server_type      6000 non-null   str    
 5   cpu_usage        5708 non-null   float64
 6   memory_usage     5691 non-null   float64
 7   disk_io          5707 non-null   float64
 8   network_latency  5705 non-null   float64
 9   error_rate       5693 non-null   float64
 10  temperature      5724 non-null   float64
 11  health_score     6000 non-null   float64
 12  alert_title      6000 non-null   str    
 13  system_log       6000 non-null   str    
 14  failure          3576 non-null   float64
 15  anomaly_score    6000 non-null   int64  
dtypes: float64(8), int64(2), str(6)
memory usage: 750.1 KB


In [5]:
train_data.isna().sum()

id                    0
timestamp             0
server_id             0
region                0
server_type           0
cpu_usage           292
memory_usage        309
disk_io             293
network_latency     295
error_rate          307
temperature         276
health_score          0
alert_title           0
system_log            0
failure            2424
anomaly_score         0
dtype: int64

In [6]:
train_data['timestamp'].value_counts()

timestamp
2025-03-01 16:57:00    14
2025-03-01 18:35:00    13
2025-03-01 05:23:00    12
2025-03-01 13:56:00    11
2025-03-01 21:18:00    11
                       ..
2025-03-01 18:11:00     1
2025-03-01 13:47:00     1
2025-03-01 09:45:00     1
2025-03-01 05:27:00     1
2025-03-01 20:41:00     1
Name: count, Length: 1415, dtype: int64

In [7]:
for name, data in [('train', train_data), ('test', test_data)]:
    ts = pd.to_datetime(data['timestamp'])
    print(name, ts.min(), ts.max(), 'days:', ts.dt.date.nunique())

train 2025-03-01 00:00:00 2025-03-01 23:59:00 days: 1
test 2025-03-15 00:01:00 2025-03-15 23:59:00 days: 1


# data cleaning

In [8]:
def impute(X, col, mean):
    newcol = []
    for i in range(len(X[col])):
        if pd.isna(X[col].iloc[i]):
            newcol.append(mean)
        else:
            newcol.append(X[col].iloc[i])
    return newcol

def onehot(data, col, categs):
  for c in categs:
    column = []
    for v in data[col]:
      if v == c:
        column.append(1)
      else:
        column.append(0)
    data[col+'_'+str(c)] = column
  data = data.drop(columns=col)
  return data

In [9]:
impcols = ['cpu_usage', 'memory_usage', 'disk_io', 'network_latency', 'error_rate', 'temperature']
means = {}
for col in impcols:
    means[col] = train_data[col].mean()

ohcols = ['region', 'server_type']
categs = {}
for col in ohcols:
    categs[col] = sorted(train_data[col].unique())

def clean(X):
    X = X.copy()
    
    for col in impcols:
        X[col] = impute(X, col, means[col])

    newcol = []
    for s in X['server_id']:
        newcol.append(int(s.split('_')[1]))
    X['server_num'] = newcol
    X = X.drop(columns=['server_id', 'id'])

    for col in ohcols:
        X = onehot(X, col, categs[col])

    X = X.drop(columns=['alert_title', 'system_log'])

    ts = pd.to_datetime(X['timestamp'])
    X['minute_of_day'] = ts.dt.hour*60 + ts.dt.minute
    X = X.drop(columns=['timestamp'])
    
    return X
    

In [10]:
alert_groups = [
    ('ok',      ['beat', 'hear', 'ping', 'sweep', 'normal', 'routin', 'check', 'chek', 'chcek', 'cehck', 'hceck', 'status', 'report', 'health']),
    ('memory',  ['ram', 'mem', 'mmeory']),
    ('error',   ['err', 'rror', 'eror', 'reror']),
    ('heat',    ['temp', 'therm', 'heat', 'hot', 'cool']),
    ('network', ['netw', 'netow', 'packet', 'connect', 'timeout', 'latenc', 'respon']),
    ('disk',    ['disk', 'storage', 'i/o', 'bottle']),
    ('cpu',     ['cpu', 'pcu', 'process', 'proecss', 'porcess', 'compute', 'cmopute', 'overload']),
]

def alert_type(title):
    s = str(title).lower().replace(' ', '')
    for name, words in alert_groups:
        for w in words:
            if w in s:
                return name
    return 'other'

types = [alert_type(t) for t in train_data['alert_title']]
print(pd.Series(types).value_counts())
for i in range(len(types)):
    if types[i] == 'other':
        print(train_data['alert_title'].iloc[i])

ok         1952
network     825
error       777
memory      690
cpu         619
heat        587
disk        527
other        23
Name: count, dtype: int64
rma utilization critical for srv_047
High ltaency alert for srv_051
Stoarge performance degraded for srv_059
Conenctivity degradation for srv_059
High latecny alert for srv_065
Tepmerature warning on srv_068
tehrmal threshold approaching for srv_077
Storgae performance degraded for srv_100
High latecny alert for srv_104
High tepmerature detected — srv_112
packte loss detected on srv_134
High latenyc alert for srv_146
Packte loss detected on srv_152
CUP usage spike detected on srv_152
HIGH LAETNCY ALERT FOR SRV_158
Conncetivity degradation for srv_174
High CUP load alert for srv_174
srv_181: CUP threshold exceeded
storgae performance degraded for srv_189
Temeprature warning on srv_190
Connecitvity degradation for srv_193
High ltaency alert for srv_194
High tmeperature detected — srv_196


In [11]:
levels = ['DEBUG', 'INFO', 'WARN', 'ERROR', 'CRIT']
events = ['thermal', 'ecc', 'oom', 'cascading', 'link flap', 'write failed',
          'unreachable', 'connection reset', 'gc pause', 'check latency',
          'retry', 'delayed', 'reallocation', 'capacity']

def log_features(log):
    lines = str(log).split('\n')
    f = {'n_lines': len(lines)}
    for lv in levels:
        f[lv] = 0
        for line in lines:
            if lv in line:             
                f[lv] += 1
    for e in events:
        f[e] = 0
        for line in lines:
            if e in line.lower():
                f[e] += 1
    return f

In [12]:


def clean2(X):
    X = X.copy()
    
    X = X.drop(columns=['server_type', 'server_id', 'id'])

    ohcols = ['region']
    for col in ohcols:
        X = onehot(X, col, categs[col])

    rows = []
    for log in X['system_log']:
        rows.append(log_features(log))
    X = pd.concat([X, pd.DataFrame(rows, index=X.index)], axis=1)
    X = X.drop(columns=['system_log'])
    
    X['alert_type'] = [alert_type(t) for t in X['alert_title']]
    X = onehot(X, 'alert_type', ['ok', 'memory', 'error', 'heat', 'network', 'disk', 'cpu', 'other'])
    X = X.drop(columns=['alert_title'])
    
    ts = pd.to_datetime(X['timestamp'])
    X['minute_of_day'] = ts.dt.hour*60 + ts.dt.minute
    X = X.drop(columns=['timestamp'])
    
    return X
    

In [13]:
X = train_data[['id', 'timestamp', 'server_id', 'region', 'server_type', 'cpu_usage', 'memory_usage', 'disk_io', 'network_latency', 'error_rate', 'temperature', 'health_score', 'alert_title', 'system_log']]
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               6000 non-null   int64  
 1   timestamp        6000 non-null   str    
 2   server_id        6000 non-null   str    
 3   region           6000 non-null   str    
 4   server_type      6000 non-null   str    
 5   cpu_usage        5708 non-null   float64
 6   memory_usage     5691 non-null   float64
 7   disk_io          5707 non-null   float64
 8   network_latency  5705 non-null   float64
 9   error_rate       5693 non-null   float64
 10  temperature      5724 non-null   float64
 11  health_score     6000 non-null   float64
 12  alert_title      6000 non-null   str    
 13  system_log       6000 non-null   str    
dtypes: float64(7), int64(1), str(6)
memory usage: 656.4 KB


In [14]:
X = clean2(X)
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cpu_usage           5708 non-null   float64
 1   memory_usage        5691 non-null   float64
 2   disk_io             5707 non-null   float64
 3   network_latency     5705 non-null   float64
 4   error_rate          5693 non-null   float64
 5   temperature         5724 non-null   float64
 6   health_score        6000 non-null   float64
 7   region_ap-east      6000 non-null   int64  
 8   region_ap-south     6000 non-null   int64  
 9   region_eu-central   6000 non-null   int64  
 10  region_eu-west      6000 non-null   int64  
 11  region_us-east      6000 non-null   int64  
 12  region_us-west      6000 non-null   int64  
 13  n_lines             6000 non-null   int64  
 14  DEBUG               6000 non-null   int64  
 15  INFO                6000 non-null   int64  
 16  WARN             

# subtask 1

## validation

In [15]:
def eval1(model, X_clean, y, metric, groups=None):
    scores = []
    kf = GroupKFold(n_splits=5)

    for fit_rows, val_rows in kf.split(X_clean, y, groups):
        X_fit = X_clean.iloc[fit_rows]
        y_fit = y.iloc[fit_rows]
        X_val = X_clean.iloc[val_rows]
        y_val = y.iloc[val_rows]
    
        model.fit(X_fit, y_fit)                  
        preds = model.predict(X_val)
        scores.append(metric(y_val, preds))

    print('folds:', [round(s, 3) for s in scores])
    print('mean:', round(np.mean(scores), 4), ' spread:', round(np.std(scores), 4))
    return scores

## model + preds

In [16]:
labeled = train_data.loc[~train_data['failure'].isna()]
X_clean = clean2(labeled[Xcols])
y = labeled['failure'].astype(int)
groups = X_clean['minute_of_day'] // 300
print(X_clean.shape, y.value_counts())

(3576, 42) failure
0    2874
1     702
Name: count, dtype: int64


In [17]:
print(X_clean.dtypes)

cpu_usage             float64
memory_usage          float64
disk_io               float64
network_latency       float64
error_rate            float64
temperature           float64
health_score          float64
region_ap-east          int64
region_ap-south         int64
region_eu-central       int64
region_eu-west          int64
region_us-east          int64
region_us-west          int64
n_lines                 int64
DEBUG                   int64
INFO                    int64
WARN                    int64
ERROR                   int64
CRIT                    int64
thermal                 int64
ecc                     int64
oom                     int64
cascading               int64
link flap               int64
write failed            int64
unreachable             int64
connection reset        int64
gc pause                int64
check latency           int64
retry                   int64
delayed                 int64
reallocation            int64
capacity                int64
alert_type

In [18]:
scores0 = eval1(DummyClassifier(strategy='constant', constant=1), X_clean, y, f1_score, groups)

scores1 = eval1(HistGradientBoostingClassifier(random_state=0), X_clean, y, f1_score, groups)

scores2 = eval1(HistGradientBoostingClassifier(class_weight='balanced', random_state=0), X_clean, y, f1_score, groups)

folds: [0.321, 0.319, 0.331, 0.35, 0.319]
mean: 0.328  spread: 0.012
folds: [0.607, 0.57, 0.592, 0.636, 0.578]
mean: 0.5965  spread: 0.0234
folds: [0.618, 0.66, 0.633, 0.658, 0.66]
mean: 0.6459  spread: 0.0171


In [19]:
model = HistGradientBoostingClassifier(class_weight='balanced', random_state=0)   # or whichever scored best
model.fit(X_clean, y)

X_test = clean2(test_data[Xcols])
pred1 = model.predict(X_test).astype(int)

# subtask 2

In [20]:
X = train_data[Xcols]
X_clean = clean2(X)
y = train_data['anomaly_score'].astype(int)
groups = X_clean['minute_of_day'] // 300
print(X_clean.shape, y.value_counts())

(6000, 42) anomaly_score
77    222
75    215
82    212
79    211
72    203
     ... 
96      2
19      2
0       1
97      1
13      1
Name: count, Length: 92, dtype: int64


In [21]:
scores0 = eval1(DummyRegressor(strategy='constant', constant=1), X_clean, y, mean_absolute_error, groups)

scores1 = eval1(HistGradientBoostingRegressor(loss='absolute_error', random_state=0), X_clean, y, mean_absolute_error, groups)

folds: [60.285, 71.134, 72.511, 72.135, 71.45]
mean: 69.503  spread: 4.6343
folds: [14.821, 4.976, 4.783, 4.953, 4.852]
mean: 6.8769  spread: 3.9724


In [22]:
model = HistGradientBoostingRegressor(loss='absolute_error', random_state=0)
model.fit(X_clean, y)

X_test = clean2(test_data[Xcols])
pred2 = model.predict(X_test)
pred2 = np.round(pred2)

# submission

In [24]:
sub1 = pd.DataFrame({'id': test_data['id'], 'subtaskID': 1, 'answer': pred1})
sub2 = pd.DataFrame({'id': test_data['id'], 'subtaskID': 2, 'answer': pred2})
sub = pd.concat([sub1, sub2], ignore_index=True)
sub.to_csv('submission.csv', index=False)
print(sub.shape, sub1['answer'].value_counts())

(3000, 3) answer
0    1177
1     323
Name: count, dtype: int64
